# Sistema de Recomendação de Músicas Nacionais com TF-IDF

### Integrantes
1. Nome: Arthur Menezes Botelho - Matrícula: 231003362
2. Nome: Lucas Saad Rodrigues   - Matrícula: 231035393


### Implementação
Este notebook implementa um sistema de recomendação baseado em **TF-IDF** (Term Frequency - Inverse Document Frequency) aplicado a músicas nacionais.

O sistema segue as seguintes etapas:

1. **Carregamento do dataset**
2. **Construção do vocabulário**: conjunto de todos os termos únicos
3. **Construção do corpus**: perfil de cada música como bag-of-attributes
4. **Cálculo do TF-IDF**: representação vetorial de cada música
5. **Construção do perfil do usuário**: a partir da matriz de utilidade
6. **Recomendação**: via similaridade de cosseno

## 1. Importações e Configurações

Importamos as bibliotecas necessárias e definimos as constantes globais do sistema.

- `SKIP_FIELDS`: campos do dataset que não entram no perfil do item (ex: nome da música)
- `MULTI_VALUE_FIELDS`: campos que podem conter múltiplos valores separados por `;` (ex: artistas)

In [638]:
import csv
from dataclasses import dataclass
from math import log
import pandas as pd 
import random

SKIP_FIELDS = ['track_name']
MULTI_VALUE_FIELDS = ['artists']

## 2. Estrutura de Dados

Definimos a classe `Document`, que representa o **perfil de um item** (música) no sistema.

Cada documento possui:
- `identifier`: nome da música
- `bag_of_attributes`: lista de termos que descrevem a música (gênero, artista, time_signature)
- `term_frequencies`: dicionário com a contagem de cada termo do vocabulário neste documento
- `tfidf_score`: dicionário com o score TF-IDF de cada termo do vocabulário para este documento

In [639]:
@dataclass
class Document:
    identifier: str
    bag_of_attributes: list[str]
    term_frequencies: dict[str, int]
    tfidf_score: dict[str, int]

## 3. Carregamento do Dataset

A função `load_dataset` lê o arquivo CSV e retorna uma lista de dicionários, onde cada dicionário representa uma música com seus atributos.

In [640]:
def load_dataset(filepath: str) -> list[dict]:
    with open(filepath, 'r') as file:
        return list(csv.DictReader(file))

## 4. Construção do Vocabulário

O **vocabulário** é o conjunto de todos os termos únicos presentes na coleção de documentos (corpus).

Ele é a base para a matriz TF-IDF: cada termo do vocabulário corresponde a uma dimensão no vetor de cada documento.

Campos em `SKIP_FIELDS` são ignorados. Campos em `MULTI_VALUE_FIELDS` são separados por `;` antes de serem adicionados ao vocabulário.

In [641]:
def build_vocabulary(data: list[dict]) -> set[str]:
    vocabulary = set()
    for document in data:
        for field, value in document.items():
            if field in SKIP_FIELDS:
                continue
            if field in MULTI_VALUE_FIELDS:
                vocabulary.update(value.split(';'))
            else:
                vocabulary.add(value)

    global VOCABULARY
    VOCABULARY = vocabulary
    
    return vocabulary

## 5. Construção do Corpus

O **corpus** é a coleção de todos os documentos. Aqui, cada música é transformada em um objeto `Document`.

O `bag_of_attributes` de cada música é construído concatenando os valores dos seus atributos em uma lista de termos — a representação textual da música que será usada pelo TF-IDF.

In [642]:
def build_corpus(data: list[dict], vocabulary: set[str]) -> list[Document]:
    corpus = []
    for document in data:
        bag_of_attributes = []
        term_frequencies = dict.fromkeys(vocabulary, 0)
        identifier = ''
        for field, value in document.items():
            if field == 'track_name':
                identifier = value
            elif field in MULTI_VALUE_FIELDS:
                bag_of_attributes.extend(value.split(';'))
            else:
                bag_of_attributes.append(value)

        tfidf_score = {}
        track = Document(identifier, bag_of_attributes, term_frequencies, tfidf_score)
        corpus.append(track)
    return corpus

## 6. Cálculo do TF-IDF

O TF-IDF é composto por duas métricas:

### 6.1 Document Frequency (DF)
Para cada termo do vocabulário, contamos em quantos documentos ele aparece.



In [643]:
def compute_document_frequencies(corpus: list[Document], vocabulary: set[str]) -> dict[str, int]:
    """Conta em quantos documentos cada termo do vocabulário aparece."""
    document_frequencies = dict.fromkeys(vocabulary, 0)
    for document in corpus:
        for term in set(document.bag_of_attributes):
            if term in vocabulary:
                document_frequencies[term] += 1

    global DOCUMENT_FREQUENCIES 
    DOCUMENT_FREQUENCIES = document_frequencies
    
    return document_frequencies


### 6.2 Term Frequency (TF)
Mede a frequência relativa de um termo dentro de um documento específico:

$$TF(t, d) = \frac{\text{contagem de } t \text{ em } d}{\text{total de termos em } d}$$



In [644]:
def compute_TF(document: Document) -> dict[str, float]:
    """Calcula o Term Frequency de cada termo para um documento específico."""
    tf_scores = {}
    for field, value in document.term_frequencies.items():
        tf_score = value / len(document.bag_of_attributes)
        tf_scores[field] = tf_score

    global TF_SCORES 
    TF_SCORES = tf_scores
    
    return tf_scores


### 6.3 Inverse Document Frequency (IDF)
Penaliza termos que aparecem em muitos documentos, valorizando os mais específicos:

$$IDF(t) = \log\left(\frac{N}{DF(t)}\right)$$

onde $N$ é o número total de documentos.



In [645]:
def compute_IDF(total_number_documents: int, document_frequencies: dict[str, int]) -> dict[str, float]:
    """Calcula o Inverse Document Frequency de cada termo no corpus."""
    IDF_scores = {}
    for field, value in document_frequencies.items():
        idf_score = log(total_number_documents / value)
        IDF_scores[field] = idf_score

    global IDF_SCORES 
    IDF_SCORES = idf_score

    return IDF_scores

### 6.4 TF-IDF
O score final é o produto das duas métricas:

$$TFIDF(t, d) = TF(t, d) \times IDF(t)$$

In [646]:
def tfidf(corpus: list[Document], document_frequencies: dict[str, int]):
    """Calcula o score TF-IDF para cada termo em cada documento do corpus."""
    total_number_documents = len(corpus)
    IDF_scores = compute_IDF(total_number_documents, document_frequencies)

    for document in corpus:
        for word in document.bag_of_attributes:
            document.term_frequencies[word] += 1

        tf_scores = compute_TF(document)

        for field, value in tf_scores.items():
            tfidf_score = value * IDF_scores[field]
            document.tfidf_score[field] = tfidf_score

## 7. Perfil do Usuário

O **perfil do usuário** é construído a partir da **matriz de utilidade**, que registra as avaliações do usuário para cada música.

Para cada música avaliada positivamente, somamos seu vetor TF-IDF ponderado pela nota do usuário:

$$\text{perfil}(u) = \sum_{i \in avaliados} rating(u, i) \times TFIDF(i)$$

O resultado é um vetor que representa o "gosto médio ponderado" do usuário, na mesma dimensão do espaço TF-IDF.

In [647]:
def build_user_profile(user_ratings: list[int], corpus: list[Document]) -> dict[str, float]:
    user_profile = {}
    for rating, document in zip(user_ratings, corpus):
        if rating:
            for word, tfidf_val in document.tfidf_score.items():
                weighted_score = tfidf_val * rating
                user_profile[word] = user_profile.get(word, 0.0) + weighted_score
    return user_profile

## 8. Similaridade de Cosseno

A **similaridade de cosseno** mede o ângulo entre dois vetores no espaço TF-IDF. Quanto menor o ângulo, mais similares são os vetores.

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

- Valor **1**: vetores idênticos (mesma direção)
- Valor **0**: vetores sem nenhuma similaridade (perpendiculares)

Usamos a similaridade de cosseno para comparar o perfil do usuário com o vetor TF-IDF de cada música não avaliada.

In [648]:
def cosine_similarity(profile_a: dict[str, float], profile_b: dict[str, float]) -> float:
    common_words = set(profile_a.keys()).intersection(set(profile_b.keys()))
    dot_product = sum(profile_a[word] * profile_b[word] for word in common_words)

    mag_a = sum(val ** 2 for val in profile_a.values()) ** 0.5
    mag_b = sum(val ** 2 for val in profile_b.values()) ** 0.5

    if mag_a == 0 or mag_b == 0:
        return 0.0

    return dot_product / (mag_a * mag_b)

## 9. Recomendação

A função `recommend_songs` gera as recomendações para um usuário:

1. Constrói o perfil do usuário a partir das suas avaliações
2. Calcula a similaridade de cosseno entre o perfil e cada música **não avaliada**
3. Retorna as `top_n` músicas com maior similaridade

In [649]:
def recommend_songs(user_ratings: list[int], corpus: list[Document], top_n: int = 15) -> list[tuple]:
    user_profile = build_user_profile(user_ratings, corpus)
    recommendations = []

    for i, document in enumerate(corpus):
        if not user_ratings[i]:
            sim_score = cosine_similarity(user_profile, document.tfidf_score)
            recommendations.append((document.identifier, sim_score))

    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:top_n]

## 10. Interface e Inicialização do Sistema

### 10.1 Geração da Matriz de Utilidade
A matriz de utilidade simula as avaliações de múltiplos usuários para todas as músicas. Cada linha representa um usuário e cada coluna uma música. Os valores variam de 0 (não avaliado) a 5 (melhor avaliação).

In [650]:
def generate_random_matrix(rows: int, cols: int, vals: range, seed: int = 1) -> list:
    """Gera uma matriz de utilidade simulada com avaliações aleatórias."""
    random.seed(seed)
    return [[random.choice(vals) for _ in range(cols)] for _ in range(rows)]

### 10.2 Cadastro de Novo Usuário
O sistema apresenta uma amostra aleatória de músicas para o novo usuário avaliar, construindo assim seu perfil inicial.

In [651]:
def register_new_user(corpus: list, num_samples: int = 5) -> list[int]:
    """Coleta avaliações do novo usuário para uma amostra de músicas."""
    print("Para iniciar, avalie as seguintes musicas com valores inteiros de 1 a 5, sendo 1 o pior e 5 o melhor nivel de avaliacao.")
    print("(Insira 0 se nao conhecer ou nao quiser avaliar a musica)\n")

    # Redefine a seed para gerar sample_indices aleatorios a cada execucao
    random.seed()

    user_ratings = [0] * len(corpus)
    sample_indices = random.sample(range(len(corpus)), num_samples)

    for idx in sample_indices:
        song = corpus[idx]
        while True:
            try:
                rating = int(input(f"Como voce avalia '{song.identifier}'? (0-5): "))
                if 0 <= rating <= 5:
                    user_ratings[idx] = rating
                    break
                else:
                    print("Insira um numero valido entre 0 e 5.")
            except ValueError:
                print("Entrada invalida. Por favor, insira um inteiro.")

    print("\nGerando perfil personalizado...\n")
    return user_ratings


### 10.3 Setup do Sistema
Inicializa todo o pipeline: carrega o dataset, constrói o vocabulário, o corpus, calcula o TF-IDF e gera a matriz de utilidade.

In [652]:
def setup_recommender(filepath: str, num_users: int = 100, seed: int = 42):
    """Inicializa o sistema de recomendação completo."""
    print("Inicializando Sistema de Recomendacao...")

    data = load_dataset(filepath)
    vocabulary = build_vocabulary(data)
    corpus = build_corpus(data, vocabulary)

    document_frequencies = compute_document_frequencies(corpus, vocabulary)
    tfidf(corpus, document_frequencies)

    num_songs = len(corpus)
    utility_matrix = generate_random_matrix(rows=num_users, cols=num_songs, vals=range(0, 6), seed=seed)
    global UTILITY_MATRIX 
    UTILITY_MATRIX = utility_matrix

    print(f"Inicializacao completa! {num_songs} musicas foram adicionadas e uma matriz de utilidade {num_users}x{num_songs} foi gerada.")
    return corpus, utility_matrix

## 11. Execução do Sistema

Aqui inicializamos o sistema com o dataset e executamos a interface de recomendação para um novo usuário.

> **Atenção:** Certifique-se de que o arquivo `dataset.csv` está no mesmo diretório que este notebook antes de executar.

In [653]:
def run_recommender_ui(corpus: list):
    print("=" * 40)
    print("--- Sistema de Recomendacao de Musicas Nacionais - IIA ---")
    print("=" * 40)

    new_user_ratings = register_new_user(corpus, num_samples=10)
    top_recommendations = recommend_songs(new_user_ratings, corpus, top_n=10)

    print("\n" + "=" * 40)
    if top_recommendations:
        print("     Recomendacoes Personalizadas para Voce:     ")
        print("=" * 40)
        for rank, (song_name, score) in enumerate(top_recommendations, start=1):
            print(f"{rank}. {song_name} (Score: {score:.4f})")
    else:
        print("Nao foi possivel gerar recomendacoes. Por favor, avalie mais musicas para melhorar seu perfil.")
    print("=" * 40)


corpus, utility_matrix = setup_recommender('dataset.csv')
run_recommender_ui(corpus)

Inicializando Sistema de Recomendacao...
Inicializacao completa! 50 musicas foram adicionadas e uma matriz de utilidade 100x50 foi gerada.
--- Sistema de Recomendacao de Musicas Nacionais - IIA ---
Para iniciar, avalie as seguintes musicas com valores inteiros de 1 a 5, sendo 1 o pior e 5 o melhor nivel de avaliacao.
(Insira 0 se nao conhecer ou nao quiser avaliar a musica)

Insira um numero valido entre 0 e 5.

Gerando perfil personalizado...


     Recomendacoes Personalizadas para Voce:     
1. Canto de Ossanha (Score: 0.5560)
2. Deixa Eu Te Querer / De Repente o Ceu (Ao Vivo) (Score: 0.3391)
3. Tem Cafe / Uma Historia Assim feat. Gaab (Ao Vivo) (Score: 0.2678)
4. Baixa Essa Guarda / Shortinho Saint-Tropez / Ponto Final / O Queiroz (Ao Vivo) (Score: 0.2279)
5. Nem Voce Nem Eu (Score: 0.2167)
6. Lalange (Score: 0.1422)
7. Flashback (Participacao Especial de Marcos & Belutti) (Ao Vivo) (Score: 0.1278)
8. Coracao na Contramao (Score: 0.1166)
9. Prazer Eu Sou Ferrugem (Ao Vivo) (Score: 

# Exemplos


## 1. Estrutura de Dados 

In [654]:
@dataclass
class Document:
    identifier: str
    bag_of_attributes: list[str]
    term_frequencies: dict[str, int]
    tfidf_score: dict[str, int]

document = corpus[0]

### 1.1 Identifier e Bag of Attributes

In [655]:
print(
    f"Track name: {document.identifier}\n"
    f"Bag of attributes: {document.bag_of_attributes}"
)

Track name: O Que Sera (A Flor da Pele)
Bag of attributes: ['milton nascimento', 'chico buarque', 'mpb', '3']


### 1.2 Term Frequency

In [656]:
df = pd.DataFrame.from_dict(document.term_frequencies, orient='index', columns=['Term Frequeny'])
print(df[df['Term Frequeny'] > 0])     

                   Term Frequeny
chico buarque                  1
mpb                            1
milton nascimento              1
3                              1


### 1.3 TF-IDF Score

In [657]:
df = pd.DataFrame.from_dict(document.tfidf_score, orient='index', columns=['TF-IDF Score'])
print(df[df['TF-IDF Score'] > 0])

                   TF-IDF Score
chico buarque          0.978006
mpb                    0.173287
milton nascimento      0.804719
3                      0.300993


## 2. Vocabulary

In [658]:
print(VOCABULARY)

{'ane alma', 'fabio jr.', 'zeze di camargo & luciano', 'marcos & belutti', 'flavio venturini', 'gusttavo lima', 'henrique & juliano', 'kleiton & kledir', 'biquini cavadao', 'elba ramalho', 'tribalistas', 'casuarina', 'moinho', 'Pixote', '1', '6', 'kevi jonny', 'aldair playboy', '7', 'seu jorge', '5', 'jimmy pratt', 'tierry', 'projeto norte', 'baden powell', 'ferrugem', 'pagode', 'inimigos da hp', 'chico buarque', 'nando reis', 'mpb', 'luiz gonzaga', 'geraldo azevedo', 'sandra de sa', 'forro', 'gaab', 'vitor fernandes', 'kid abelha', 'milton nascimento', 'nacao zumbi', 'diogo nogueira', 'legiao urbana', 'samba de raiz', 'jorge aragao', '3', 'tarcisio do acordeon', 'priscila senna', 'grupo menos e mais', 'guitar', 'chico rey & parana', 'sertanejo', 'mr. dan', 'jorge & mateus', 'rodriguinho', 'turma do pagode', 'ednardo', 'alceu valenca', 'chico science', 'diego & victor hugo', 'roupa nova', '4', 'matheus & kauan', 'jorge vercillo', 'calcinha preta', 'liniker', 'nattan'}


## 3. Document Frequency

In [659]:
df = pd.DataFrame.from_dict(DOCUMENT_FREQUENCIES, orient='index', columns=['Count'])
print(df.nlargest(10, 'Count'))

                 Count
mpb                 25
4                   19
3                   15
pagode               8
sertanejo            8
5                    6
forro                6
6                    5
turma do pagode      4
1                    3


## 4. Term Frequency (TF)

In [660]:
df = pd.DataFrame.from_dict(TF_SCORES, orient='index', columns=['Scores'])
print(df.nlargest(10, 'Scores'))

                             Scores
Pixote                     0.333333
sertanejo                  0.333333
4                          0.333333
ane alma                   0.000000
fabio jr.                  0.000000
zeze di camargo & luciano  0.000000
marcos & belutti           0.000000
flavio venturini           0.000000
gusttavo lima              0.000000
henrique & juliano         0.000000


## 5. Matriz de Utilidade


In [661]:
df = pd.DataFrame(utility_matrix)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 10)
print(df)

    0   1   2   3   4   ...  45  46  47  48  49
0    5   0   0   5   2  ...   0   3   0   2   2
1    4   2   0   5   3  ...   1   1   3   3   2
2    5   5   4   1   5  ...   5   1   5   3   4
3    0   3   3   4   3  ...   0   0   2   2   1
4    0   1   4   0   0  ...   0   5   5   0   1
..  ..  ..  ..  ..  ..  ...  ..  ..  ..  ..  ..
95   3   5   3   4   2  ...   4   2   2   1   2
96   5   5   0   2   0  ...   4   4   3   4   0
97   0   4   5   2   4  ...   0   4   5   4   1
98   2   5   5   1   5  ...   4   4   4   0   3
99   1   4   0   5   3  ...   3   0   0   4   1

[100 rows x 50 columns]
